# 6. JSON configuration files and the GUI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/06_json_and_cli.ipynb)

Everything we did in Python can also be stored in a single **JSON configuration file** that holds the
equipment, the plant, the analysis settings, and the output options. The same file runs from Python, opens in
the OpenPyTEA GUI, and runs from the command line.
When you publish a paper or report with OpenPyTEA, we highly recommend sharing this file as supplementary material, so that readers can reproduce every result.

This short demo leads into the GUI demonstration. Run the cells as we go; there is no task.

## Learning goals

- Read an OpenPyTEA JSON configuration file and map it to the Python objects.
- Run a study from a file with `run_openpytea()` and reload its results with `load_results()`.
- Save a scenario as a JSON file that the GUI can open.

## 0) Setup

In [ ]:
# Run this cell. It installs OpenPyTEA if it is missing (e.g., on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
# Make sure the example configuration files are available (downloaded from
# GitHub on Colab)
import os, urllib.request

os.makedirs("assets", exist_ok=True)
for name in ["smr_hydrogen.json", "we_hydrogen.json"]:
    if not os.path.exists(f"assets/{name}"):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/pbtamarona/openpytea_workshop/"
            f"main/notebooks/assets/{name}",
            f"assets/{name}")
print(sorted(os.listdir("assets")))

## 1) Anatomy of a configuration file

A combined file has four top-level blocks:

| Block | Content | Python equivalent |
|----------------|------------------------------------------------------|------------------------------|
| `equipment` | list of items, each with the `Equipment` arguments | notebook 01 |
| `plant` | the `Plant` configuration dictionary, uncertainty blocks included | notebooks 02 and 04 |
| `analysis` | which analyses to run and their arguments | notebook 03 and 04 |
| `output` | whether to save JSON results and plots | - |

`assets/smr_hydrogen.json` is the SMR plant of notebook 05 written this way:

In [ ]:
import json

with open("assets/smr_hydrogen.json") as f:
    smr_config = json.load(f)

print("Top-level blocks:", list(smr_config))
print("\nFirst equipment item:",
      json.dumps(smr_config["equipment"][0], indent=2))
print("\nAnalysis block:", json.dumps(smr_config["analysis"],
                                      indent=2)[:500], "...")

## 2) Running a study from a file

`run_openpytea()` builds the equipment and the plant, runs every analysis with `"run": true`, and writes the
results to the output directory. It returns the results as a dictionary.

In [ ]:
from openpytea import run_openpytea, load_results

results = run_openpytea("assets/smr_hydrogen.json", output_dir="outputs/smr")
print("Result blocks:", list(results))
written = sorted(os.listdir("outputs/smr"))
for ext in (".json", ".png"):
    files = [f for f in written if f.endswith(ext)]
    print(f"{len(files)} {ext} files written, e.g.", files[:2])

In [ ]:
# Results are plain JSON, so they can be reloaded later without recomputing
from openpytea.plotting import plot_monte_carlo, plot_tornado

saved = load_results("outputs/smr/Steam reforming_analysis_results.json")
fig, ax = plot_monte_carlo(saved["monte_carlo"], metric="LCOP", bins=40)
fig, ax = plot_tornado(saved["tornado"])

## 3) A scenario file for the GUI

A scenario is an edited copy of a configuration file. Below we give the electrolysis plant cheap power (0.02 USD/kWh) and save it under a new name. The same JSON files can be opened in the OpenPyTEA GUI (`Open project`), which loads their equipment and plant settings. We will use them in the GUI demo.

In [ ]:
with open("assets/we_hydrogen.json") as f:
    we_config = json.load(f)

we_config["plant"]["plant_name"] = "Water electrolysis, cheap power"
we_config["plant"]["variable_opex_inputs"]["electricity"].update({
    "price": 0.02,
    "price_uncertainty": {"std": 0.005, "min": 0.005, "max": 0.05}})

os.makedirs("outputs", exist_ok=True)
with open("outputs/we_cheap_power.json", "w") as f:
    json.dump(we_config, f, indent=2)

In [ ]:
from openpytea.plotting import plot_multiple_monte_carlo

cheap = run_openpytea("outputs/we_cheap_power.json",
                      output_dir="outputs/we_cheap")
fig, ax = plot_multiple_monte_carlo(
    [saved["monte_carlo"], cheap["monte_carlo"]], metric="LCOP", bins=40,
    label="Levelized cost of hydrogen / [USD/kg]")

## 4) The command-line interface

Installing OpenPyTEA also installs an `openpytea` command, which runs the same file from a terminal without
any Python code:

```bash
openpytea run assets/we_hydrogen.json --output-dir outputs/we
```

Because it needs no notebook or Python script, it is handy for running many scenario files in a batch
script, on a remote server or computing cluster, or as a step in an automated workflow.

`openpytea --help` lists all commands. The [Config Files & CLI](https://openpytea.readthedocs.io/en/latest/user_guide/io_workflow.html) page of the documentation describes
the file format and the commands in detail.

## Recap

- One JSON file holds a complete, reproducible study.
- The same file runs with `run_openpytea()`, opens in the GUI, and runs with `openpytea run`.
- `load_results()` reloads results for plotting without recomputing.

Thank you for taking part in the workshop. Questions and contributions are welcome at
https://github.com/pbtamarona/OpenPyTEA.

## References

These are the references used across the six notebooks. The notebooks that use each one are in brackets.

- Acevedo, Y.M., Prosser, J.H., Huya-Kouadio, J.M., McNamara, K.R., & James, B.D. (2023). *Hydrogen Production Cost with Alkaline Electrolysis*. DOE-SA-09629-1, Strategic Analysis Inc. https://doi.org/10.2172/2203367 (Notebook 5: water electrolysis data)
- Boyano, A., Blanco-Marigorta, A.M., Morosuk, T., & Tsatsaronis, G. (2011). Exergoenvironmental analysis of a steam methane reforming process for hydrogen production. *Energy, 36*(4), 2202-2214. https://doi.org/10.1016/j.energy.2010.05.020 (Notebook 5: steam methane reforming data)
- Tamarona, P.B., Vlugt, T.J.H., & Ramdin, M. (2026). OpenPyTEA: An open-source python toolkit for techno-economic assessment of chemical process plants and energy systems with economic sensitivity and uncertainty evaluation. *SoftwareX, 35*, 102816. https://doi.org/10.1016/j.softx.2026.102816 (Notebook 1: software architecture in section 2; Notebooks 3 to 6)
- Towler, G., & Sinnott, R. (2022). *Chemical Engineering Design* (3rd ed.). Elsevier. https://doi.org/10.1016/C2019-0-02025-0 (Notebook 1: chapter 7, capital cost estimating; Notebook 2: chapters 7 to 9)
- Turton, R., Shaeiwitz, J.A., Bhattacharyya, D., & Whiting, W.B. (2018). *Analysis, Synthesis, and Design of Chemical Processes* (5th ed.). Prentice Hall. (Notebook 1: appendix A, cost correlations; Notebook 2: chapters 7 to 10)
- Van der Spek, M., Roussanaly, S., & Rubin, E.S. (2019). Best practices and recent advances in CCS cost engineering and economic analysis. *IJGGC, 83*, 91-104. https://doi.org/10.1016/j.ijggc.2019.02.006 (Notebooks 3 and 4)
- OpenPyTEA documentation: https://openpytea.readthedocs.io (all notebooks; Notebook 6: User Guide, Config Files & CLI page)